<a href="https://colab.research.google.com/github/late-cat/Sec_colab/blob/main/notebooks/01_model_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Customer Churn Optimization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

## loading dataset

In [ ]:
url = "https://raw.githubusercontent.com/late-cat/Sec_colab/main/data/Churn_Modelling.csv"
dataset = pd.read_csv(url)
dataset.head()

## dividing data set

In [ ]:
X = dataset.drop(['RowNumber', 'CustomerId', 'Surname', 'Exited'], axis=1)
Y = dataset['Exited']

In [ ]:
X.head()

In [ ]:
Y

## Feature engeneering

In [ ]:
geography = pd.get_dummies(X['Geography'], drop_first=True)
gender = pd.get_dummies(X['Gender'], drop_first=True)

In [ ]:
X = X.drop(['Geography', 'Gender'], axis=1)

In [ ]:
X = pd.concat([X, geography, gender], axis=1)

In [ ]:
X.head()

## test train split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

## fetaure scaling

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## handling imbalanced data (SMOTE)

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, Y_train_sm = smote.fit_resample(X_train, Y_train)

In [ ]:
Y_train_sm.value_counts()

## training base model & cross-validation

In [ ]:
base_rf = RandomForestClassifier(random_state=42)
base_rf.fit(X_train_sm, Y_train_sm)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_cv = cross_val_score(base_rf, X_train_sm, Y_train_sm, cv=skf, scoring='f1')
print(f"Base CV F1: {base_cv.mean():.4f} ± {base_cv.std():.4f}")

## hyperparameter tuning

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

In [ ]:
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train_sm, Y_train_sm)

In [ ]:
tuned_rf = random_search.best_estimator_
random_search.best_params_

## deep error analysis & comparison

In [ ]:
base_preds = base_rf.predict(X_test)
base_probs = base_rf.predict_proba(X_test)[:, 1]

tuned_preds = tuned_rf.predict(X_test)
tuned_probs = tuned_rf.predict_proba(X_test)[:, 1]

In [ ]:
tuned_cv = cross_val_score(tuned_rf, X_train_sm, Y_train_sm, cv=skf, scoring='f1')
print(f"Tuned CV F1: {tuned_cv.mean():.4f} ± {tuned_cv.std():.4f}")

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Base Model': [
        accuracy_score(Y_test, base_preds),
        precision_score(Y_test, base_preds),
        recall_score(Y_test, base_preds),
        f1_score(Y_test, base_preds),
        roc_auc_score(Y_test, base_probs)
    ],
    'Tuned Model': [
        accuracy_score(Y_test, tuned_preds),
        precision_score(Y_test, tuned_preds),
        recall_score(Y_test, tuned_preds),
        f1_score(Y_test, tuned_preds),
        roc_auc_score(Y_test, tuned_probs)
    ]
})
comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(confusion_matrix(Y_test, tuned_preds), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')

RocCurveDisplay.from_estimator(tuned_rf, X_test, Y_test, ax=axes[1])
axes[1].set_title('ROC-AUC')

PrecisionRecallDisplay.from_estimator(tuned_rf, X_test, Y_test, ax=axes[2])
axes[2].set_title('PR-AUC')

plt.tight_layout()
plt.show()

## saving the model

In [ ]:
save_dir = '../models/optimized models'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(tuned_rf, os.path.join(save_dir, 'final_model.joblib'))
joblib.dump(sc, os.path.join(save_dir, 'scaler.joblib'))